# Generator Dataset Latih SFT Telko Indonesia V2 (23-Tool Candidate Pool + Persona Multi-Turn)\n### Dataset Latih Agentic BSS TelkomNusa V2 (Persiapan Khusus Multi-Turn & Long Context 4096)\n\nNotebook ini menghasilkan 3.200 data latih berkualitas tinggi dengan perbaikan arsitektural total:\n1. Injeksi 23 Tool API BSS ke Seluruh Sampel: Setiap sampel kini menyertakan katalog lengkap 23 tool agar model terlatih melakukan seleksi diskriminatif 23-way, identik dengan skenario benchmark held-out.\n2. Injeksi Persona Extraction (Multi-Turn & Single-Turn): 400 sampel baru di mana persona pengguna (nomor HP, nama, tanggal lahir, paket aktif) sudah ada dalam system prompt. Model dilatih untuk langsung mengekstrak parameter dari persona dan mengeksekusi tool tanpa bertanya ulang ke user.\n3. Dataset Bitext Telco Terlokalisasi: 2.200 sampel percakapan alami (Billing, Payment, Usage, Roaming, Add-on, Subscriptions, General Support).\n4. Injeksi Ekosistem BSS Lengkap: Family BSS, Refill Coupons, Transaksi Konten Digital, dan Roaming.\n5. Format Standar Qwen2.5 ChatML Tool-Calling: Mendukung MAX_SEQ_LENGTH = 4096 tanpa pemotongan konteks.\n6. One-Click Export: Membagi menjadi train_sft_telko_indonesia.jsonl (90%) dan val_sft_telko_indonesia.jsonl (10%) serta membuat file ZIP unduh otomatis.

## Section 0: Setup Dependencies

In [ ]:
!pip install -q --no-warn-conflicts datasets openai requests tqdm pandas

In [ ]:
import os
import re
import json
import time
import random
import shutil
import urllib.parse
import urllib.request
from typing import List, Dict, Any, Optional, Tuple
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

# Set seed untuk reproduktifitas penelitian
random.seed(42)
print(" Dependencies & Environment Siap!")

## Section 1: Definisi Lengkap 23 Skema API BSS TelkomNusa

In [ ]:
# Daftar Lengkap 23 Skema Tool API BSS TelkomNusa Standar OpenAI/Qwen Format
TELKOMNUSA_BSS_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "GET__health",
            "description": "Pemeriksaan status kesehatan sistem layanan (Health Check).",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_add-on-subscriptions",
            "description": "[Add-on_0001] Menampilkan daftar Layanan Tambahan (Add-on) yang sedang aktif pada nomor pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan (contoh: 081234567890)"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "DELETE__rate_add-on-subscriptions",
            "description": "Membatalkan atau berhenti berlangganan Layanan Tambahan (Add-on).",
            "parameters": {
                "type": "object",
                "properties": {
                    "prodId": {"type": "string", "description": "ID Produk Layanan Tambahan (contoh: AD00000003)"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["prodId", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_content-purchases",
            "description": "[Bill_0001] Menampilkan rincian biaya penggunaan konten dan layanan digital pada rentang tanggal tertentu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"},
                    "startDt": {"type": "string", "description": "Tanggal awal pencarian (YYYYMMDD)"},
                    "endDt": {"type": "string", "description": "Tanggal akhir pencarian (YYYYMMDD)"}
                },
                "required": ["svcMgmtNum", "startDt", "endDt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_current-bills",
            "description": "[Bill_0002] Menampilkan rincian Tagihan Berjalan (real-time) nomor pascabayar bulan ini.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pascabayar pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_current-data-limits",
            "description": "[Usage_0001] Menampilkan total kuota data, kuota terpakai, dan sisa kuota data saat ini.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__plan",
            "description": "[Prod_0001] Menampilkan informasi daftar paket tarif, harga langganan, dan produk yang tersedia.",
            "parameters": {
                "type": "object",
                "properties": {
                    "prodId": {"type": "string", "description": "ID Produk Paket Tarif"},
                    "minProdAmt": {"type": "integer", "description": "Batas minimum harga paket (IDR)"},
                    "maxProdAmt": {"type": "integer", "description": "Batas maksimum harga paket (IDR)"},
                    "minProdAge": {"type": "integer", "description": "Batas minimum usia pelanggan"},
                    "maxProdAge": {"type": "integer", "description": "Batas maksimum usia pelanggan"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_family_bills",
            "description": "[Family_0003] Menampilkan tagihan berjalan anggota keluarga yang terdaftar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}
                },
                "required": ["familySvcMgmtNum", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_family_data-usage",
            "description": "[Family_0001] Menampilkan sisa kuota data anggota keluarga yang terdaftar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}
                },
                "required": ["familySvcMgmtNum", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_family_real-time-bill",
            "description": "[Family_0008] Menampilkan tagihan berjalan (real-time) nomor anggota keluarga.",
            "parameters": {
                "type": "object",
                "properties": {
                    "familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}
                },
                "required": ["familySvcMgmtNum", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_family_info",
            "description": "[Family_0009] Menampilkan informasi dan relasi anggota keluarga yang terdaftar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_fixed-bills",
            "description": "[Bill_0003] Menampilkan Tagihan Bulanan Resmi yang telah dikonfirmasi/terbit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"},
                    "startDt": {"type": "string", "description": "Tanggal awal periode tagihan (YYYYMMDD)"},
                    "endDt": {"type": "string", "description": "Tanggal akhir periode tagihan (YYYYMMDD)"}
                },
                "required": ["svcMgmtNum", "startDt", "endDt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_mobile-payments",
            "description": "[Payment_0001] Menampilkan riwayat transaksi Layanan Potong Pulsa pada periode tertentu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"},
                    "startDt": {"type": "string", "description": "Tanggal awal pencarian (YYYYMMDD)"},
                    "endDt": {"type": "string", "description": "Tanggal akhir pencarian (YYYYMMDD)"}
                },
                "required": ["svcMgmtNum", "startDt", "endDt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__plan_remained-contracts",
            "description": "[Plan_0003] Menampilkan sisa masa kontrak berlangganan paket atau perangkat HP.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__plan_roaming",
            "description": "[Roaming_0001] Menampilkan daftar paket roaming internasional yang tersedia untuk negara tujuan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "countryCode": {"type": "string", "description": "Kode Negara (ISO 2-huruf: ID, SG, MY, JP, US, SA)"},
                    "countryName": {"type": "string", "description": "Nama Negara Tujuan (contoh: Singapura, Jepang, Arab Saudi)"}
                },
                "required": ["countryCode"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "POST__plan_roaming_subscriptions",
            "description": "[Roaming_0005] Mendaftarkan atau mengaktifkan paket roaming internasional untuk nomor pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "roamingProdId": {"type": "string", "description": "ID Produk Paket Roaming (contoh: RM00000007)"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["roamingProdId", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "DELETE__plan_roaming_subscriptions",
            "description": "[Roaming_0002] Membatalkan atau menonaktifkan paket roaming yang sedang aktif.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_unpaid-bills",
            "description": "[Bill_0001] Menampilkan rincian Tunggakan Tagihan yang belum dibayar oleh pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "POST__rate_unpaid-bills_payment",
            "description": "[Bill_0002] Melakukan pembayaran untuk melunasi Tunggakan Tagihan pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__rate_refill-coupons",
            "description": "Menampilkan ketersediaan dan status Kupon Extra Kuota (Refill Coupon) pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "POST__rate_refill-coupons_use",
            "description": "Menggunakan atau mengaktifkan Kupon Extra Kuota (jenis: data atau voice).",
            "parameters": {
                "type": "object",
                "properties": {
                    "rfilTrgtltm": {"type": "string", "description": "Jenis kuota yang ditambah (data / voice)"},
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["rfilTrgtltm", "svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "DELETE__rate_auto-payment",
            "description": "Membatalkan atau menonaktifkan layanan pembayaran debit otomatis (Auto-Payment).",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "GET__info",
            "description": "[Persona_0001] Menampilkan informasi profil persona dan paket aktif pelanggan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}
                },
                "required": ["svcMgmtNum"]
            }
        }
    }
]

print(f" Terdaftar {len(TELKOMNUSA_BSS_TOOLS)} Skema API BSS TelkomNusa!")

## Section 2: AI Localization Engine (Bluesminds GPT-5-Mini + Direct Fallback)

In [ ]:
BLUESMINDS_KEY = "YOUR_API_KEY_HERE"
from openai import OpenAI

ai_client = OpenAI(api_key=BLUESMINDS_KEY, base_url="https://api.bluesminds.com/v1")

def direct_google_translate_en_id(text: str) -> str:
    """Fallback penerjemah langsung EN -> ID anti-crash"""
    try:
        url = "https://translate.googleapis.com/translate_a/single?client=gtx&sl=en&tl=id&dt=t&q=" + urllib.parse.quote(text)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            res = json.loads(response.read().decode('utf-8'))
            return "".join([seg[0] for seg in res[0] if seg[0]])
    except Exception:
        return text

def batch_localize_telecom_queries(queries: List[str]) -> List[str]:
    """Mengubah pertanyaan bahasa Inggris Bitext menjadi bahasa Indonesia telko alami"""
    system_p = """Anda adalah pakar bahasa telekomunikasi Indonesia (TelkomNusa / Telkomsel / Indosat). 
Tugas Anda: Ubah daftar keluhan/pertanyaan pelanggan bahasa Inggris berikut ke Bahasa Indonesia alami, luwes, dan sesuai kebiasaan orang Indonesia saat bertanya ke Customer Service telekomunikasi (bisa sedikit santai, gaul, atau sopan).

Contoh:
- 'there are charges on my bill that i dont recognize' -> 'Min, ada tagihan aneh di nomor saya yang gak saya kenal nih.'
- 'my internet bill is incorrect, can you help me dispute it?' -> 'Tagihan internet saya bulan ini kok kayaknya salah hitung ya, bisa dibantu cek?'
- 'activate roaming for my trip' -> 'Tolong aktifkan paket roaming untuk perjalanan saya.'

Kembalikan HANYA JSON array of string ["kalimat_1", "kalimat_2", ...] dengan jumlah item PERSIS SAMA!"""
    
    try:
        resp = ai_client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": system_p},
                {"role": "user", "content": json.dumps(queries, ensure_ascii=False)}
            ],
            temperature=0.3,
            timeout=25
        )
        txt = resp.choices[0].message.content.strip()
        if txt.startswith("```"):
            txt = re.sub(r"^```(?:json)?|```$", "", txt, flags=re.MULTILINE).strip()
        parsed = json.loads(txt)
        if isinstance(parsed, dict):
            for k, v in parsed.items():
                if isinstance(v, list):
                    parsed = v
                    break
        if isinstance(parsed, list) and len(parsed) == len(queries):
            return [str(x) for x in parsed]
    except Exception:
        pass
    
    # Fallback Direct Engine
    return [direct_google_translate_en_id(q) for q in queries]

print(" AI Localization Engine Siap!")

## Section 3: Download & Stratified Sampling Dataset Bitext

In [ ]:
print(" Mengunduh dataset 'bitext/Bitext-telco-llm-chatbot-training-dataset' dari Hugging Face...")
raw_ds = load_dataset("bitext/Bitext-telco-llm-chatbot-training-dataset", split="train")
df_raw = pd.DataFrame(raw_ds)

print(f" Total Data Bitext Asli: {len(df_raw)} baris")
print(f" Daftar Kategori: {df_raw['category'].unique().tolist()}")
print(f" Total Intent: {df_raw['intent'].nunique()} Intent")

# Stratified Sampling: Ambil maks 85 sampel terunik per intent (~2.200 sampel)
SAMPLES_PER_INTENT = 85
sampled_rows = []

for intent, group in df_raw.groupby('intent'):
    n_take = min(len(group), SAMPLES_PER_INTENT)
    sampled_rows.append(group.sample(n=n_take, random_state=42))
    
df_sampled = pd.concat(sampled_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\n Berhasil mengekstrak {len(df_sampled)} sampel terkurasi dari Bitext!")

## Section 4: Engine Transformasi Bitext ke Qwen ChatML Tool Calling

In [ ]:
def generate_dummy_phone():
    prefix = random.choice(["0811", "0812", "0813", "0821", "0822", "0852", "0853", "0881"])
    suffix = "".join([str(random.randint(0, 9)) for _ in range(7)])
    return f"{prefix}{suffix}"

SYSTEM_PROMPT = "Anda adalah asisten AI resmi Customer Care TelkomNusa. Tugas Anda adalah membantu pelanggan mengecek tagihan, kuota data, paket roaming, kupon extra kuota, paket keluarga, dan layanan telekomunikasi lainnya secara ramah, sopan, dan akurat. Gunakan fungsi alat (tools) yang tersedia jika diperlukan."

def synthesize_bitext_record(row_id: int, indo_query: str, intent: str, category: str) -> Dict[str, Any]:
    phone = generate_dummy_phone()
    is_multi_turn = random.random() < 0.25  # 25% probabilitas Multi-Turn Slot Filling
    
    # 1. Kasus BILLING
    if intent in ["dispute_invoice", "invoices"]:
        tool_name = "GET__rate_current-bills" if "berjalan" in indo_query.lower() or random.random() < 0.5 else "GET__rate_unpaid-bills"
        tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
        
        if is_multi_turn:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": indo_query},
                {"role": "assistant", "content": "Halo Kak! Boleh dibantu informasikan nomor handphone TelkomNusa yang ingin dicek rincian tagihannya?", "tool_calls": None},
                {"role": "user", "content": f"Nomor saya {phone}"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps({"svcMgmtNum": phone})}}]},
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "billAmt": 145000, "status": "Tercatat", "period": "Juni 2025"})},
                {"role": "assistant", "content": f"Terima kasih Kak! Berdasarkan sistem kami untuk nomor {phone}, tercatat tagihan sebesar **Rp145.000** untuk periode Juni 2025. Ada yang ingin ditanyakan lagi terkait rincian ini?"}
            ]
        else:
            user_q = f"{indo_query} (Nomor saya {phone})"
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_q},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps({"svcMgmtNum": phone})}}]},
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "billAmt": 175000, "status": "Belum Lunas", "dueDate": "2025-06-25"})},
                {"role": "assistant", "content": f"Halo Kak! Tagihan untuk nomor TelkomNusa {phone} adalah sebesar **Rp175.000** dengan tanggal jatuh tempo 25 Juni 2025. Apakah ingin kami bantu pandu pembayarannya?"}
            ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "billing", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # 2. Kasus CONSUMPTION
    elif intent in ["check_usage", "check_excess_data_charges", "set_usage_limits"]:
        tool_name = "GET__rate_current-data-limits"
        tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
        user_q = f"{indo_query} Nomor saya {phone}"
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_q},
            {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps({"svcMgmtNum": phone})}}]},
            {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "remDataAmt": "18.2 GB", "totOfrDataAmt": "35 GB", "activeUntil": "2025-07-20"})},
            {"role": "assistant", "content": f"Halo Kak! Sisa kuota data untuk nomor {phone} saat ini adalah **18.2 GB** dari total kuota 35 GB, aktif sampai dengan tanggal 20 Juli 2025. Ada lagi yang bisa kami bantu?"}
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "consumption", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # 3. Kasus PAYMENT
    elif intent in ["check_mobile_payments", "pay", "payment_methods", "schedule_payments"]:
        tool_name = "GET__rate_mobile-payments" if intent == "check_mobile_payments" else "POST__rate_unpaid-bills_payment"
        tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
        
        if tool_name == "GET__rate_mobile-payments":
            args = {"svcMgmtNum": phone, "startDt": "20250601", "endDt": "20250617"}
            ret = {"resultCode": "0000", "purchaseList": [{"service": "Google Play Store", "amount": 45000, "date": "2025-06-10"}]}
            resp_txt = f"Berdasarkan data sistem pada nomor {phone}, terdapat 1 transaksi potong pulsa sebesar Rp45.000 untuk pembelian layanan Google Play Store pada tanggal 10 Juni 2025."
        else:
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "paymentStatus": "Sukses", "paidAmt": 150000}
            resp_txt = f"Pembayaran tagihan sebesar Rp150.000 untuk nomor {phone} telah **berhasil diproses**. Terima kasih telah melunasi tagihan Anda tepat waktu!"
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{indo_query} (Nomor: {phone})"},
            {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps(args)}}]},
            {"role": "tool", "name": tool_name, "content": json.dumps(ret)},
            {"role": "assistant", "content": resp_txt}
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "payment", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # 4. Kasus SERVICES
    elif intent in ["activate_roaming", "activate_call_management_services", "deactivate_call_management_services"]:
        if intent == "activate_roaming":
            tool_name = "GET__plan_roaming"
            tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{indo_query} Saya mau ke Singapura."},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__plan_roaming", "arguments": json.dumps({"countryCode": "SG", "countryName": "Singapura"})}}]},
                {"role": "tool", "name": "GET__plan_roaming", "content": json.dumps({"resultCode": "0000", "packages": [{"prodId": "RM00000007", "name": "Paket Roaming Passport Asia 7 Hari (10GB)", "price": 150000}]})},
                {"role": "assistant", "content": "Halo Kak! Untuk perjalanan ke Singapura, tersedia **Paket Roaming Passport Asia 7 Hari (10GB)** dengan tarif Rp150.000. Apakah ingin langsung kami bantu aktifkan pada nomor Anda?"}
            ]
        else:
            tool_name = "DELETE__rate_add-on-subscriptions"
            tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{indo_query} Nomor saya {phone}"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps({"prodId": "AD00000003", "svcMgmtNum": phone})}}]},
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "status": "Sukses Berhenti Berlangganan"})},
                {"role": "assistant", "content": f"Layanan tambahan (Add-on) pada nomor {phone} telah **berhasil dinonaktifkan**. Anda tidak akan dikenakan biaya perpanjangan berikutnya."}
            ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "services", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # 5. Kasus SUBSCRIPTION
    elif intent in ["cancel_plan", "change_plan", "check_cancellation_fee", "sign_up_for_plan"]:
        tool_name = "GET__plan" if "cari" in indo_query.lower() or "rekomendasi" in indo_query.lower() else "GET__plan_remained-contracts"
        tool_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == tool_name]
        
        if tool_name == "GET__plan":
            args = {"minProdAmt": 50000, "maxProdAmt": 100000}
            ret = {"resultCode": "0000", "plans": [{"prodId": "PL001", "name": "TelkomNusa Super Max 50GB", "price": 85000}]}
            resp_txt = "Kami merekomendasikan paket **TelkomNusa Super Max 50GB** seharga Rp85.000/bulan yang sudah termasuk kuota utama 50GB serta telepon sepuasnya sesama TelkomNusa."
        else:
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "contractStatus": "Aktif", "remainedMonths": 4, "penaltyFee": 0}
            resp_txt = f"Masa kontrak berlangganan untuk nomor {phone} masih tersisa **4 bulan lagi** tanpa biaya penalti jika diselesaikan hingga masa akhir kontrak."
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{indo_query} (Nomor: {phone})"},
            {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": json.dumps(args)}}]},
            {"role": "tool", "name": tool_name, "content": json.dumps(ret)},
            {"role": "assistant", "content": resp_txt}
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "subscription", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # 6. Kasus COMPLAINTS & General Chat (Out-of-Domain)
    else:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": indo_query},
            {"role": "assistant", "content": "Halo Kak! Kami mohon maaf atas ketidaknyamanan yang dialami. Keluhan Anda telah kami catat untuk diteruskan ke tim optimasi jaringan kami. Untuk pengecekan kendala lebih lanjut, pastikan perangkat telah di-restart atau hubungi call center 188. Ada layanan lain yang bisa kami bantu?", "tool_calls": None}
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "general_support", "tools": [], "messages": messages}

print(" Synthesis Bitext Engine Siap!")

## ‍‍‍ Section 5: Generator Khusus 23 BSS Tools Lengkap (Family, Kupon, Konten & Roaming Eksekusi)

In [ ]:
def generate_dedicated_bss_samples(start_id: int) -> List[Dict[str, Any]]:
    """
    Menghasilkan ~600 data latih khusus untuk menjamin 100% dari 23 Tool BSS TelkomNusa
    memiliki data latih yang presisi dan berkualitas tinggi.
    """
    samples = []
    cur_id = start_id
    
    # 1. KELOMPOK PAKET KELUARGA (Family BSS Tools) - 150 Sampel
    family_relations = ["anak", "istri", "suami", "orang tua", "adik", "kakak"]
    for _ in range(150):
        main_phone = generate_dummy_phone()
        fam_phone = generate_dummy_phone()
        rel = random.choice(family_relations)
        sub_case = random.choice(["usage", "bill", "realtime_bill", "info"])
        
        if sub_case == "usage":
            t_name = "GET__rate_family_data-usage"
            q = random.choice([
                f"Min, tolong cek sisa kuota data {rel} saya di nomor {fam_phone}, nomor utama saya {main_phone}.",
                f"Berapa sisa kuota internet anggota keluarga nomor {fam_phone}? Nomor saya {main_phone}.",
                f"Tolong periksa sisa kuota keluarga nomor {fam_phone} yang terdaftar di akun {main_phone}."
            ])
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "remDataAmt": "12.4 GB", "totDataAmt": "20 GB"}
            resp = f"Halo Kak! Sisa kuota data keluarga untuk nomor {fam_phone} ({rel}) saat ini adalah **12.4 GB** dari total 20 GB."
            
        elif sub_case == "bill":
            t_name = "GET__rate_family_bills"
            q = random.choice([
                f"Tolong cek rincian tagihan berjalan {rel} saya {fam_phone}. Nomor pengelola saya {main_phone}.",
                f"Bisa bantu lihat tagihan anggota keluarga nomor {fam_phone}? Akun utama: {main_phone}."
            ])
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "familyBillAmt": 125000, "period": "Juni 2025"}
            resp = f"Tagihan berjalan untuk nomor anggota keluarga {fam_phone} periode Juni 2025 tercatat sebesar **Rp125.000**."
            
        elif sub_case == "realtime_bill":
            t_name = "GET__rate_family_real-time-bill"
            q = random.choice([
                f"Mohon periksa tagihan real-time nomor {fam_phone} yang terdaftar sebagai keluarga dari {main_phone}.",
                f"Berapa tagihan berjalan saat ini untuk nomor {rel} saya {fam_phone}? Nomor saya {main_phone}."
            ])
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "realtimeBill": 98000, "status": "Aktif"}
            resp = f"Tagihan real-time untuk nomor keluarga {fam_phone} saat ini adalah sebesar **Rp98.000**."
            
        else:
            t_name = "GET__rate_family_info"
            q = random.choice([
                f"Tolong tampilkan daftar anggota keluarga yang terdaftar di nomor layanan saya {main_phone}.",
                f"Bisa cek siapa saja anggota keluarga yang terhubung ke nomor {main_phone}?",
                f"Mohon konfirmasi data keluarga pada nomor utama {main_phone}."
            ])
            args = {"svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "familyList": [{"phone": fam_phone, "relation": rel, "status": "Terhubung"}]}
            resp = f"Pada nomor utama {main_phone}, terdapat 1 anggota keluarga terdaftar yaitu nomor {fam_phone} sebagai **{rel}**."
            
        t_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == t_name]
        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "family_bss",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": t_name, "arguments": json.dumps(args)}}]},
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp}
            ]
        })
        cur_id += 1
        
    # 2. KELOMPOK KUPON EXTRA KUOTA (Refill Coupons) - 150 Sampel
    for _ in range(150):
        phone = generate_dummy_phone()
        is_use = random.random() < 0.5
        
        if is_use:
            t_name = "POST__rate_refill-coupons_use"
            item_type = random.choice(["data", "voice"])
            q = random.choice([
                f"Saya mau gunakan Kupon Extra Kuota {item_type} untuk nomor {phone}.",
                f"Tolong aktifkan kupon refill {item_type} pada nomor saya {phone}.",
                f"Pakai 1 kupon extra {item_type} untuk nomor {phone} sekarang."
            ])
            args = {"rfilTrgtltm": item_type, "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "resultMessage": "Sukses", "appliedType": item_type}
            resp = f"Kupon Extra Kuota jenis **{item_type}** telah **berhasil diaktifkan** pada nomor {phone}. Tambahan kuota sudah masuk ke akun Anda."
        else:
            t_name = "GET__rate_refill-coupons"
            q = random.choice([
                f"Mohon periksa apakah nomor {phone} memiliki Kupon Extra Kuota yang bisa digunakan?",
                f"Cek ketersediaan kupon refill kuota untuk nomor saya {phone}.",
                f"Apakah ada kupon kuota gratis atau refill kuota di nomor {phone}?"
            ])
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "availableCoupons": 2, "couponType": ["data", "voice"]}
            resp = f"Halo Kak! Nomor {phone} memiliki **2 Kupon Extra Kuota** yang siap digunakan (tersedia untuk kuota data atau telepon). Apakah ingin digunakan sekarang?"
            
        t_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == t_name]
        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "refill_coupons",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": t_name, "arguments": json.dumps(args)}}]},
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp}
            ]
        })
        cur_id += 1
        
    # 3. KELOMPOK TRANSAKSI DIGITAL & AUTO-PAYMENT - 150 Sampel
    for _ in range(150):
        phone = generate_dummy_phone()
        sub_case = random.choice(["content", "auto_pay_del", "fixed_bill", "addon_list"])
        
        if sub_case == "content":
            t_name = "GET__rate_content-purchases"
            q = f"Tolong tampilkan rincian biaya pembelian konten digital pada nomor {phone} periode 1 Mei 2025 sampai 31 Mei 2025."
            args = {"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "contentList": [{"name": "Langganan Netflix Premium", "amount": 186000, "date": "20250515"}]}
            resp = f"Berdasarkan data sistem pada nomor {phone}, terdapat transaksi pembelian konten **Netflix Premium** sebesar Rp186.000 pada tanggal 15 Mei 2025."
            
        elif sub_case == "auto_pay_del":
            t_name = "DELETE__rate_auto-payment"
            q = random.choice([
                f"Saya mau membatalkan pengaturan pembayaran debit otomatis (auto-payment) untuk nomor {phone}.",
                f"Tolong hentikan layanan auto debit tagihan nomor {phone} saya.",
                f"Nonaktifkan penarikan otomatis di kartu {phone} sekarang."
            ])
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Sukses Dibatalkan"}
            resp = f"Layanan debit otomatis (Auto-Payment) untuk nomor {phone} telah **berhasil dinonaktifkan**. Anda dapat membayar tagihan secara manual melalui aplikasi MyTelkomNusa."
            
        elif sub_case == "fixed_bill":
            t_name = "GET__rate_fixed-bills"
            q = f"Mohon periksa Tagihan Bulanan Resmi yang sudah terbit untuk nomor {phone} dari tanggal 01 April 2025 sampai 30 April 2025."
            args = {"startDt": "20250401", "endDt": "20250430", "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "fixedBill": 210000, "status": "Lunas"}
            resp = f"Tagihan Bulanan Resmi nomor {phone} periode April 2025 adalah sebesar **Rp210.000** (status: Lunas)."
            
        else:
            t_name = "GET__rate_add-on-subscriptions"
            q = f"Mohon tampilkan daftar Layanan Tambahan (Add-on) yang saat ini aktif pada nomor {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "addOnList": [{"prodId": "AD00000003", "name": "Nada Sambung Pribadi (NSP)", "price": 9000}]}
            resp = f"Pada nomor {phone}, saat ini terdapat 1 Layanan Tambahan aktif yaitu **Nada Sambung Pribadi (NSP)** seharga Rp9.000/bulan."
            
        t_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == t_name]
        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "digital_transactions",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": t_name, "arguments": json.dumps(args)}}]},
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp}
            ]
        })
        cur_id += 1
        
    # 4. KELOMPOK ROAMING SUBSCRIPTIONS & PERSONA PROFIL - 150 Sampel
    for _ in range(150):
        phone = generate_dummy_phone()
        sub_case = random.choice(["roam_sub", "roam_del", "info_persona", "health"])
        
        if sub_case == "roam_sub":
            t_name = "POST__plan_roaming_subscriptions"
            prod_id = random.choice(["RM00000007", "RM00000010", "RM00000014"])
            q = random.choice([
                f"Tolong daftarkan paket roaming {prod_id} untuk nomor {phone} perjalanan ke luar negeri.",
                f"Aktifkan paket roaming {prod_id} di kartu saya {phone} sekarang.",
                f"Saya mau beli dan langsung aktifkan paket roaming {prod_id} untuk nomor {phone}."
            ])
            args = {"roamingProdId": prod_id, "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Berhasil Terdaftar", "validUntil": "2025-07-01"}
            resp = f"Paket Roaming Passport ({prod_id}) telah **berhasil diaktifkan** untuk nomor {phone}. Paket siap digunakan saat tiba di negara tujuan."
            
        elif sub_case == "roam_del":
            t_name = "DELETE__plan_roaming_subscriptions"
            q = random.choice([
                f"Karena saya sudah kembali ke Indonesia, tolong batalkan paket roaming nomor {phone}.",
                f"Hentikan langganan paket roaming internasional untuk nomor {phone}.",
                f"Mohon nonaktifkan paket roaming pada nomor {phone}."
            ])
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Paket Roaming Sukses Diberhentikan"}
            resp = f"Paket roaming pada nomor {phone} telah **berhasil dinonaktifkan**. Anda tidak akan dikenakan biaya roaming lagi."
            
        elif sub_case == "info_persona":
            t_name = "GET__info"
            q = f"Tolong tampilkan profil persona dan informasi akun pelanggan untuk nomor layanan {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "customerName": "Ahmad Fauzi", "tier": "TelkomNusa Priority Gold", "status": "Aktif"}
            resp = f"Informasi akun nomor {phone}: Pelanggan atas nama **Ahmad Fauzi**, status kartu Aktif dengan tier keanggotaan **TelkomNusa Priority Gold**."
            
        else:
            t_name = "GET__health"
            q = "Mohon periksa status kesehatan server dan kesiapan sistem BSS TelkomNusa saat ini."
            args = {}
            ret = {"resultCode": "0000", "status": "UP", "database": "Connected", "latency": "12ms"}
            resp = "Status sistem BSS TelkomNusa saat ini berada dalam kondisi **Sehat dan Normal (UP)** dengan latensi 12ms."
            
        t_schema = [t for t in TELKOMNUSA_BSS_TOOLS if t["function"]["name"] == t_name]
        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "roaming_and_ops",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": t_name, "arguments": json.dumps(args)}}]},
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp}
            ]
        })
        cur_id += 1
        
    return samples

print(" Generator Dedicated 23 BSS Tools Siap!")

## Section 5B: Generator Khusus Persona Extraction & Multi-Turn Trajectories (400 Sampel)\n

In [ ]:
# SECTION 5B: GENERATOR PERSONA EXTRACTION & MULTI-TURN TRAJECTORIES
# Menghasilkan 400 sampel khusus di mana nomor telepon dan data pelanggan
# sudah tersedia di System Prompt Persona, melatih model untuk LANGSUNG
# mengeksekusi tool tanpa bertanya nomor telepon ke pengguna.

def generate_persona_multiturn_samples(start_id: int) -> List[Dict[str, Any]]:
    samples = []
    cur_id = start_id
    
    first_names = ["Budi", "Siti", "Ahmad", "Dewi", "Rizky", "Putri", "Hendra", "Nur", "Agus", "Lestari", "Eko", "Maya", "Dian", "Fajar", "Indah"]
    last_names = ["Santoso", "Wijaya", "Kusuma", "Pratama", "Saputra", "Siregar", "Nasution", "Hidayat", "Setiawan", "Utami", "Nugroho"]
    active_plans = [
        ("PL00000001", "Paket TelkomNusa Halo 5G Unlimited", 150000),
        ("PL00000002", "Paket Super Seru 50GB", 100000),
        ("PL00000003", "Paket Juara Internet 30GB", 75000),
        ("PL00000004", "Paket Roaming Prioritas Max", 200000)
    ]
    addons_pool = [
        ("AD00000001", "Ekstra Kuota Nonton 10GB", 25000),
        ("AD00000002", "Vidio Platinum 30 Hari", 35000),
        ("AD00000003", "Disney+ Hotstar Monthly", 39000),
        ("AD00000004", "TelkomNusa Music MAX", 15000),
        ("AD00000005", "Nada Sambung Pribadi (NSP)", 5000)
    ]

    for _ in range(400):
        phone = generate_dummy_phone()
        fam_phone = generate_dummy_phone()
        cust_name = f"{random.choice(first_names)} {random.choice(last_names)}"
        fam_name = f"{random.choice(first_names)} {random.choice(last_names)}"
        birth_year = random.randint(1975, 2004)
        birth_month = f"{random.randint(1, 12):02d}"
        birth_day = f"{random.randint(1, 28):02d}"
        birth_dt = f"{birth_year}{birth_month}{birth_day}"
        gender = random.choice(["Pria", "Wanita"])
        plan_id, plan_name, plan_fee = random.choice(active_plans)
        addon_id, addon_name, addon_fee = random.choice(addons_pool)
        
        persona = {
            "svc_num": phone,
            "cust_nm": cust_name,
            "birth_dt": birth_dt,
            "gender": gender,
            "active_plan": plan_name,
            "family_members": [{"svc_num": fam_phone, "cust_nm": fam_name, "hubungan": "keluarga"}]
        }
        
        system_content = (
            "Anda adalah asisten AI resmi Customer Care TelkomNusa. Tugas Anda adalah membantu pelanggan "
            "mengecek tagihan, sisa kuota, transaksi konten digital, paket roaming, kupon kuota, dan operasional BSS lainnya "
            "secara ramah, sopan, dan akurat.\n"
            f"Persona penggunanya adalah: {json.dumps(persona, ensure_ascii=False)}\n"
            "Aturan Respons:\n"
            "1. PENTING: Jika informasi pelanggan (seperti nomor telepon svc_num atau nama) sudah tersedia dalam data persona di atas, "
            "Anda WAJIB langsung menggunakannya sebagai argumen pemanggilan tool tanpa menanyakan kembali kepada pelanggan.\n"
            "2. Tanggal saat ini adalah 17 Juni 2025.\n"
            "3. Panggil fungsi yang relevan secara tepat menggunakan format <tool_call>.\n"
            "4. Setelah menerima hasil tool, jawab pertanyaan pelanggan dengan ramah dan informatif."
        )
        
        scenario_type = random.choice(["bill_and_addon", "usage_and_payment", "bill_and_content", "roaming_and_plan"])
        
        if scenario_type == "bill_and_addon":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Halo, bisakah Anda memeriksa apakah ada tagihan yang belum dibayar pada akun saya?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_unpaid-bills", "arguments": json.dumps({"svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_unpaid-bills", "content": json.dumps({"resultCode": "0000", "unpaidBill": {"hasUnpaid": True, "billAmt": 128500, "dueDate": "20250625"}})},
                {"role": "assistant", "content": f"Halo Kak {cust_name}! Berdasarkan sistem kami untuk nomor {phone}, saat ini terdapat tagihan yang belum dibayar sebesar **Rp128.500** dengan jatuh tempo 25 Juni 2025. Ada yang bisa kami bantu terkait tagihan ini?"},
                {"role": "user", "content": "Baik, lalu layanan tambahan (add-on) apa saja yang saat ini sedang aktif di nomor saya?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_add-on-subscriptions", "arguments": json.dumps({"svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_add-on-subscriptions", "content": json.dumps({"resultCode": "0000", "addOnList": [{"prodId": addon_id, "prodNm": addon_name, "prodAmt": addon_fee}]})},
                {"role": "assistant", "content": f"Untuk nomor Kak {cust_name}, layanan tambahan yang saat ini aktif adalah **{addon_name}** (ID: {addon_id}) dengan biaya langganan Rp{addon_fee:,}/bulan."},
                {"role": "user", "content": f"Tolong batalkan langganan layanan {addon_name} tersebut ya."},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "DELETE__rate_add-on-subscriptions", "arguments": json.dumps({"prodId": addon_id, "svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "DELETE__rate_add-on-subscriptions", "content": json.dumps({"resultCode": "0000", "cancelResult": "Sukses", "message": f"Layanan {addon_name} berhasil dihentikan"})},
                {"role": "assistant", "content": f"Baik Kak {cust_name}, langganan **{addon_name}** pada nomor {phone} telah berhasil dibatalkan dan tidak akan diperpanjang untuk periode berikutnya. Apakah ada hal lain yang bisa kami bantu?"}
            ]
            
        elif scenario_type == "usage_and_payment":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Tolong cek sisa kuota data internet saya saat ini dong."},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_current-data-limits", "arguments": json.dumps({"svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_current-data-limits", "content": json.dumps({"resultCode": "0000", "totalData": "50 GB", "usedData": "23.4 GB", "remData": "26.6 GB", "expiryDate": "20250715"})},
                {"role": "assistant", "content": f"Halo Kak {cust_name}! Sisa kuota data untuk nomor {phone} saat ini adalah **26.6 GB** dari total kuota 50 GB, aktif hingga tanggal 15 Juli 2025."},
                {"role": "user", "content": "Apakah selama bulan Juni ini ada transaksi pembayaran kecil atau potong pulsa?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_mobile-payments", "arguments": json.dumps({"startDt": "20250601", "endDt": "20250617", "svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_mobile-payments", "content": json.dumps({"resultCode": "0000", "totalAmount": 35000, "purchaseList": [{"service": "Google Play", "amount": 35000, "date": "20250610"}]})},
                {"role": "assistant", "content": f"Untuk periode 1-17 Juni 2025 pada nomor {phone}, tercatat 1 transaksi pembayaran kecil sebesar **Rp35.000** untuk pembelian Google Play pada 10 Juni 2025."}
            ]
            
        elif scenario_type == "bill_and_content":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Berapa total biaya pembelian konten digital saya pada bulan Mei lalu?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_content-purchases", "arguments": json.dumps({"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_content-purchases", "content": json.dumps({"resultCode": "0000", "contentList": [{"contentNm": "Langganan Streaming Film", "purchaseAmt": 54000}]})},
                {"role": "assistant", "content": f"Halo Kak {cust_name}! Biaya konten digital untuk nomor {phone} pada bulan Mei 2025 tercatat sebesar **Rp54.000** (Langganan Streaming Film)."},
                {"role": "user", "content": "Bisakah saya memeriksa seluruh tagihan tetap yang sudah dikonfirmasi untuk bulan Mei tersebut?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__rate_fixed-bills", "arguments": json.dumps({"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone})}}]},
                {"role": "tool", "name": "GET__rate_fixed-bills", "content": json.dumps({"resultCode": "0000", "totalBillAmt": 165000, "billDetailList": [{"billNm": "Paket Halo Pascabayar", "billAmt": 150000}, {"billNm": "PPN 11%", "billAmt": 15000}]})},
                {"role": "assistant", "content": f"Rincian tagihan tetap resmi untuk bulan Mei 2025 pada nomor {phone} adalah sebesar **Rp165.000** (termasuk PPN 11%). Ada pertanyaan lain seputar tagihan Kak?"}
            ]
            
        else: # roaming_and_plan
            age = 2025 - birth_year
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Saya berencana bepergian ke Jepang minggu depan, paket roaming apa yang tersedia?"},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__plan_roaming", "arguments": json.dumps({"countryCode": "JP", "countryName": "Jepang"})}}]},
                {"role": "tool", "name": "GET__plan_roaming", "content": json.dumps({"resultCode": "0000", "packages": [{"prodId": "RM00000002", "name": "Paket Roaming Asia Pass 7 Hari (10GB)", "price": 175000}]})},
                {"role": "assistant", "content": f"Halo Kak {cust_name}! Untuk perjalanan ke Jepang, tersedia **Paket Roaming Asia Pass 7 Hari (10GB)** seharga **Rp175.000**. Apakah ingin langsung kami bantu aktifkan di nomor {phone}?"},
                {"role": "user", "content": "Selain itu, tolong rekomendasikan paket internet yang cocok untuk kelompok usia saya."},
                {"role": "assistant", "content": None, "tool_calls": [{"type": "function", "function": {"name": "GET__plan", "arguments": json.dumps({"minProdAge": str(max(15, age - 5)), "maxProdAge": str(age + 5)})}}]},
                {"role": "tool", "name": "GET__plan", "content": json.dumps({"resultCode": "0000", "products": [{"prodId": "NA00000005", "name": "Paket Juara Muda 50GB", "prodAmt": 90000}]})},
                {"role": "assistant", "content": f"Berdasarkan kelompok usia Kak {cust_name} ({age} tahun), paket yang sangat cocok adalah **Paket Juara Muda 50GB** dengan tarif **Rp90.000/bulan**."}
            ]
            
        samples.append({
            "id": f"sft_telko_persona_{cur_id:05d}",
            "category": "persona_multiturn",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": messages
        })
        cur_id += 1

    print(f"Berhasil menghasilkan {len(samples)} sampel Persona Multi-Turn Trajectories!")
    return samples

## Section 6: Eksekusi Penggabungan & Sintesis Total (~2.800 Sampel)

In [ ]:
print(f"[*] Tahap 1: Memproses {len(df_sampled)} data Bitext terlokalisasi...\n")

batch_size = 25
final_sft_records = []

all_instructions = df_sampled["instruction"].tolist()
all_intents = df_sampled["intent"].tolist()
all_categories = df_sampled["category"].tolist()

for b_i in tqdm(range(0, len(all_instructions), batch_size), desc="Processing Bitext Samples"):
    b_raw = all_instructions[b_i:b_i+batch_size]
    b_intents = all_intents[b_i:b_i+batch_size]
    b_cats = all_categories[b_i:b_i+batch_size]
    
    b_indo = batch_localize_telecom_queries(b_raw)
    
    for idx_in_b in range(len(b_raw)):
        global_idx = b_i + idx_in_b
        rec = synthesize_bitext_record(
            row_id=global_idx + 1,
            indo_query=b_indo[idx_in_b],
            intent=b_intents[idx_in_b],
            category=b_cats[idx_in_b]
        )
        final_sft_records.append(rec)

print(f"\n[*] Tahap 2: Menghasilkan data khusus untuk seluruh 23 Tool BSS (Family, Refill, Content, Roaming Eksekusi)...")
dedicated_records = generate_dedicated_bss_samples(start_id=len(final_sft_records) + 1)
final_sft_records.extend(dedicated_records)

print(f"\n[*] Tahap 3: Menghasilkan 400 sampel Persona Multi-Turn Extraction...")
persona_records = generate_persona_multiturn_samples(start_id=len(final_sft_records) + 1)
final_sft_records.extend(persona_records)

print(f"\n Selesai. Total {len(final_sft_records)} sampel data latih SFT (100% mencakup 23 Tool API BSS) telah terbuat!")

## Section 7: Audit Tool Coverage & Split Train/Val JSONL

In [ ]:
# Verifikasi Cakupan 23 Tool API
tool_counts = {}
for r in final_sft_records:
    for msg in r["messages"]:
        if msg.get("tool_calls"):
            for tc in msg["tool_calls"]:
                tname = tc["function"]["name"]
                tool_counts[tname] = tool_counts.get(tname, 0) + 1

print("==================================================")
print(f" LAPORAN CAKUPAN TOOL DATA LATIH ({len(tool_counts)} DARI 23 TOOL TERCAKUP):")
print("==================================================")
for t in sorted(tool_counts.keys()):
    print(f"• {t:<36} : {tool_counts[t]} sampel")
print("==================================================")

# Acak dan bagi Train (90%) dan Validation (10%)
random.shuffle(final_sft_records)
split_idx = int(len(final_sft_records) * 0.9)
train_data = final_sft_records[:split_idx]
val_data = final_sft_records[split_idx:]

out_dir = "/kaggle/working/dataset_sft_telko_indonesia" if os.path.exists("/kaggle/working") else "dataset_sft_telko_indonesia"
os.makedirs(out_dir, exist_ok=True)

train_file = os.path.join(out_dir, "train_sft_telko_indonesia.jsonl")
val_file = os.path.join(out_dir, "val_sft_telko_indonesia.jsonl")

with open(train_file, "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(val_file, "w", encoding="utf-8") as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n File Train Disimpan : {train_file} ({len(train_data)} baris, {os.path.getsize(train_file)/1024:.1f} KB)")
print(f" File Val Disimpan   : {val_file} ({len(val_data)} baris, {os.path.getsize(val_file)/1024:.1f} KB)")

## Section 8: Export Archive

In [ ]:
zip_filename = "dataset_sft_telko_indonesia"
zip_dest = "/kaggle/working/" + zip_filename if os.path.exists("/kaggle/working") else zip_filename

print(f" Mengompres folder '{out_dir}' ke '{zip_dest}.zip'...")
shutil.make_archive(zip_dest, 'zip', out_dir)
zip_size_mb = os.path.getsize(f"{zip_dest}.zip") / (1024 * 1024)
print(f" File ZIP Siap: {zip_filename}.zip ({zip_size_mb:.2f} MB)\n")

# Buat Link Unduh Langsung Otomatis
print("⏳ Sedang membuat Link Download Langsung Publik (file.io)...")
import requests
try:
    with open(f"{zip_dest}.zip", "rb") as f:
        res = requests.post("https://file.io", files={"file": f})
        dlink = res.json().get("link")
        print("\n==================================================")
        print(f" Download link:\n {dlink}")
        print("==================================================")
except Exception as e:
    print(f"Link generator fallback: Unduh melalui tab Output Kaggle.")